# 10 · OCR cache cho ViTextVQA — chuẩn bị OCR-enhanced prompting (W8, bước 4)

**Vấn đề:** Error audit W7 cho thấy trên ViTextVQA, lỗi của B2 tập trung ở khâu đọc chữ: đọc chữ sai 28%, sai vùng 23%, sai dấu thanh 8%. Ablation "có / không OCR-enhanced prompting" lại là yêu cầu bắt buộc của đề cương. Để chạy được ablation này, mọi run (cả eval lẫn train) phải dùng **cùng một bản OCR**. Bản đó được tạo **một lần** ở notebook này và lưu thành file cache trên Drive.

**Notebook này làm 2 phần:**
- **Phần A (chạy ngay):** cài PaddleOCR, chạy thử 100 ảnh để đo tốc độ, rồi OCR **dev300** bằng **2 backend** (`paddleocr` và `paddledet_vietocr`). Cuối phần A có bảng chỉ số thay thế "đáp án có nằm trong OCR không".
- **Phần B (chạy SAU bước 5):** OCR **test2000** và **tập con train 8.000 câu** bằng backend đã chọn ở bước 5.

**Runtime:** chọn **T4 GPU** (Runtime → Change runtime type → T4 GPU). OCR không cần L4. Cache chỉ tạo một lần, mọi run sau đều đọc lại đúng file đó, nên GPU dùng để OCR không ảnh hưởng tính so sánh.

**Không cần** cài `transformers` hay nạp Qwen ở đây. Thư viện OCR chạy trong phiên riêng này để không đụng tới môi trường chạy VLM.

## 1. Thiết lập môi trường

Gắn Google Drive, lấy code mới nhất từ GitHub, in loại GPU.

*Kỳ vọng: commit mới nhất (có `src/eval/ocr_coverage.py`), `GPU: Tesla T4` (hoặc GPU khác, đều được), `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('src/eval/ocr_coverage.py'), 'Code cũ -> git pull chưa lấy được commit W08 bước 4'

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU (OCR sẽ chạy CPU, chậm hơn)')
print('✅ Thiết lập xong')

## 2. Cài PaddleOCR 3.7 (thử GPU trước, không được thì tự chuyển CPU)

- `paddleocr==3.7.0`: ghim phiên bản, vì code trong `src/data/ocr.py` viết theo API 3.x.
- **Thử** `paddlepaddle-gpu` từ kho của Paddle (bản CUDA 12.6). Sau khi cài sẽ kiểm tra ở **tiến trình riêng**: Paddle phải thấy GPU, và **torch vẫn thấy GPU** (VietOCR ở mục 6 cần torch).
- Nếu bước kiểm tra GPU hỏng thì cell **tự gỡ** bản GPU và cài bản CPU từ PyPI. Bản CPU luôn chạy được, chỉ chậm hơn.

*Kỳ vọng: in phiên bản `paddleocr 3.7.0`, phiên bản `paddle`, rồi `DEVICE = gpu` (hoặc `cpu` kèm lý do). Mất khoảng 2–4 phút.*

In [ ]:
import subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1200:]

code_, log = sh('pip install -q paddleocr==3.7.0')
assert code_ == 0, log
code_, log = sh('pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/')
gpu_ok = code_ == 0 and sh('python -c "import paddle; paddle.utils.run_check(); '
                           'assert paddle.device.is_compiled_with_cuda()"')[0] == 0
torch_ok = sh('python -c "import torch; assert torch.cuda.is_available()"')[0] == 0
if gpu_ok and torch_ok:
    DEVICE = 'gpu'
else:
    print('⚠ Paddle GPU không dùng được' if not gpu_ok else '⚠ Cài Paddle GPU làm torch mất GPU',
          '-> chuyển sang Paddle CPU')
    sh('pip uninstall -y -q paddlepaddle-gpu')
    code_, log = sh('pip install -q paddlepaddle')
    assert code_ == 0, log
    DEVICE = 'cpu'
print(sh('python -c "import paddleocr, paddle; print(\'paddleocr\', paddleocr.__version__, \'| paddle\', paddle.__version__)"')[1])
print('torch thấy GPU:', sh('python -c "import torch; print(torch.cuda.is_available())"')[1].strip())
print('DEVICE =', DEVICE)

## 3. Ảnh + danh sách ảnh cần OCR

Giải nén ảnh ViTextVQA từ zip trên Drive về `/content`, giống notebook 02/08. Sau đó đếm số **ảnh duy nhất** của từng file dữ liệu: nhiều câu hỏi có thể dùng chung một ảnh, và OCR chỉ cần chạy một lần cho mỗi ảnh.

*Kỳ vọng: bảng 3 dòng `dev300`, `test2000`, `train_sub8k`. Dòng `train_sub8k` phải có **2670** ảnh (khớp manifest bước 3). Không có dòng `⚠ thiếu`. Lần đầu giải nén mất khoảng 3–5 phút.*

In [ ]:
from src.eval.run_baseline import prepare_images
from src.data.ocr import images_of, default_cache_path

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
DATA_DIR = f'{DRIVE_ROOT}/data/{DS}'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
IMG = prepare_images(DS, DRIVE_ROOT)

FILES = {'dev300': 'dev_subset300_seed42.json',
         'test2000': 'test_subset2000_seed42.json',
         'train_sub8k': 'train_subset8000_byimage_seed42.json'}
IMGS = {}
for k, f in FILES.items():
    IMGS[k] = images_of([f'{DATA_DIR}/{f}'], IMG)
    miss = [p for p in IMGS[k].values() if not os.path.exists(p)]
    print(f'{k:12s} {len(IMGS[k]):5d} ảnh' + (f'  ⚠ thiếu {len(miss)} ảnh' if miss else ''))
assert len(IMGS['train_sub8k']) == 2670, 'Tập con train không khớp manifest bước 3'
ALL_B = set(IMGS['test2000']) | set(IMGS['train_sub8k'])
print(f'Phần B (test2000 ∪ train_sub8k): {len(ALL_B)} ảnh')

## 4. Chạy thử 100 ảnh (backend `paddleocr`) → đo tốc độ, xem tận mắt

Chỉ OCR 100 ảnh đầu của dev300, rồi:
- tính **giây/ảnh** và **thời gian ước tính** cho phần còn lại;
- in 3 câu hỏi kèm đáp án gold và các dòng OCR đọc được, để **xem tận mắt** OCR có đọc ra chữ tiếng Việt có dấu không.

Ảnh ViTextVQA đều có chữ, nên nếu dưới 50% ảnh đọc ra chữ thì gần như chắc chắn có lỗi cài đặt. Khi đó cell sẽ dừng.

*Kỳ vọng: lần đầu tải model OCR (vài chục MB). Sau đó `[OCR] xong … s/ảnh`. GPU khoảng 0,1–0,5 s/ảnh, CPU khoảng 1–4 s/ảnh. Tỉ lệ ảnh có chữ ≥ 50%, ví dụ in ra có dấu tiếng Việt.*

In [ ]:
import json
from src.data.ocr import load_ocr_cache

C_PADDLE = default_cache_path(DRIVE_ROOT, DS, 'paddleocr')
!python -u -m src.data.ocr --dataset {DS} --backend paddleocr --device {DEVICE} --image_dir {IMG} --files dev_subset300_seed42.json --limit_images 100

c = load_ocr_cache(C_PADDLE)
t = c['timing']; spi = t['sec'] / t['images']
with_text = sum(bool(e['lines']) for e in c['images'].values()) / len(c['images'])
print(f"\nengine: {c['engine_info']}")
print(f'{spi:.2f} s/ảnh | {100 * with_text:.0f}% ảnh có chữ')
print(f'Ước tính: phần còn lại dev300 ≈ {spi * (len(IMGS["dev300"]) - t["images"]) / 60:.0f} phút | '
      f'phần B ({len(ALL_B)} ảnh) ≈ {spi * len(ALL_B) / 60:.0f} phút')
assert with_text >= 0.5, 'Dưới 50% ảnh có chữ -> OCR cài sai? Dán log cell này cho Claude'

dev = json.load(open(f'{DATA_DIR}/{FILES["dev300"]}', encoding='utf-8'))
shown = 0
for r in dev:
    e = c['images'].get(r['image'])
    if e and shown < 3:
        shown += 1
        print(f"\n— {r['question']}  | gold: {r['answers']}")
        for ln in e['lines'][:8]:
            print(f"   {ln['conf']:.2f}  {ln['text']}")

## 5. OCR hết dev300 bằng `paddleocr`

Chạy tiếp từ chỗ dừng: 100 ảnh đã OCR ở mục 4 được giữ nguyên, không chạy lại. Nếu Colab bị ngắt giữa chừng thì chạy lại đúng cell này, vì cache được lưu sau mỗi 50 ảnh.

*Kỳ vọng: `[OCR] … sẽ OCR <còn lại>` rồi `[OCR] xong. Cache: …/ocr/paddleocr.json`.*

In [ ]:
!python -u -m src.data.ocr --dataset {DS} --backend paddleocr --device {DEVICE} --image_dir {IMG} --files dev_subset300_seed42.json

## 6. Backend thứ hai: PaddleOCR dò chữ + VietOCR đọc chữ (dev300)

VietOCR là mô hình đọc chữ **huấn luyện riêng cho tiếng Việt**. Backend này dùng **cùng model dò chữ** với `paddleocr` và chỉ thay phần đọc, nên so hai backend là so đúng một biến: phần đọc chữ.

Cài `vietocr` với `--no-deps`, vì gói trên PyPI khoá `pillow==10.2.0` và sẽ hạ cấp Pillow của Colab. Phần đọc chữ chỉ cần thêm `gdown` và `einops`.

*Kỳ vọng: `VietOCR import OK`, lần đầu tải trọng số VietOCR, rồi `[OCR] xong. Cache: …/ocr/paddledet_vietocr.json`. Chậm hơn `paddleocr` vì đọc từng hộp chữ.*

In [ ]:
code_, log = sh('pip install -q --no-deps vietocr==0.3.13 && pip install -q gdown einops')
assert code_ == 0, log
print(sh('python -c "from vietocr.tool.predictor import Predictor; print(\'VietOCR import OK\')"')[1])
C_VIET = default_cache_path(DRIVE_ROOT, DS, 'paddledet_vietocr')
!python -u -m src.data.ocr --dataset {DS} --backend paddledet_vietocr --device {DEVICE} --image_dir {IMG} --files dev_subset300_seed42.json

## 7. Chỉ số thay thế trên dev300: đáp án gold có nằm trong văn bản OCR không?

Chưa có nhãn chữ thật nên không tính được tỉ lệ lỗi ký tự của OCR. Thay vào đó, mỗi backend được đo trên 300 câu dev:
- **đáp án∈OCR %**: đáp án gold xuất hiện nguyên cụm từ trong văn bản OCR. Đây là **trần trên** cho lợi ích của OCR-enhanced prompting.
- **…sau cắt %**: như trên nhưng chỉ tính phần thực sự vào prompt (cắt ở 600 ký tự).
- **…bỏ dấu %**: bỏ dấu cả hai phía. Phần chênh so với cột đầu là phần OCR **đọc đúng chữ nhưng sai dấu thanh**.
- **Ghép cặp McNemar**: hai backend khác nhau có ý nghĩa thống kê không.

Đây chỉ là **chẩn đoán**. Backend được chọn theo luật đặt trước ở bước 5 (EM của B1 + OCR trên dev300).

*Kỳ vọng: bảng 2 dòng + 1 dòng `Ghép cặp …`, ghi ra `experiments/W08_ocr_dev300_coverage/ocr_coverage.json`.*

In [ ]:
OUT7 = f'{EXP_ROOT}/W08_ocr_dev300_coverage/ocr_coverage.json'
!python -m src.eval.ocr_coverage --data {DATA_DIR}/dev_subset300_seed42.json --caches {C_PADDLE} {C_VIET} --out {OUT7}

## 8. ⏸ DỪNG — hết Phần A

Gửi Claude: **ảnh chụp output mục 4 và mục 7** (hoặc copy chữ). Sau đó **Disconnect and delete runtime** để không tốn thêm compute.

Bước 5 (phiên L4 khác, notebook riêng) sẽ chạy B1 + OCR trên dev300 cho từng backend và template để chọn cấu hình theo luật đặt trước. Có kết quả bước 5 rồi mới quay lại chạy **Phần B** bên dưới.

## 9. PHẦN B — OCR test2000 + tập con train bằng backend đã chọn

Điền `BACKEND` theo kết quả bước 5 (`'paddleocr'` hoặc `'paddledet_vietocr'`). Khi chạy Phần B trong **phiên mới**, chạy lại mục 1 → 3 trước, rồi chạy mục 6 nếu backend là `paddledet_vietocr`. Cache dev300 đã có trên Drive nên không phải chạy lại mục 4–7.

*Kỳ vọng: 2 lượt `[OCR] xong`, thời gian gần với ước tính ở mục 4. Chạy lại khi bị ngắt sẽ tự tiếp từ chỗ dừng.*

In [ ]:
BACKEND = None          # <- điền sau bước 5, vd 'paddleocr'
assert BACKEND in ('paddleocr', 'paddledet_vietocr'), 'Điền BACKEND theo kết quả bước 5'
!python -u -m src.data.ocr --dataset {DS} --backend {BACKEND} --device {DEVICE} --image_dir {IMG} --files test_subset2000_seed42.json
!python -u -m src.data.ocr --dataset {DS} --backend {BACKEND} --device {DEVICE} --image_dir {IMG} --files train_subset8000_byimage_seed42.json

## 10. Biên bản cache (để tái lập)

Ghi sha256 của từng file cache, engine, số ảnh và thời gian OCR ra `experiments/W08_ocr_cache/ocr_cache_manifest.json`. File này nhỏ, sẽ được đưa lên git. Bản thân các file cache nằm trên Drive.

*Kỳ vọng: mỗi cache 1 dòng (backend, số ảnh, s/ảnh, sha256 12 ký tự đầu). Tập ảnh dev300/test2000/train_sub8k đều đủ với backend đã chọn.*

In [ ]:
import hashlib, datetime
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

man = {'what': 'OCR cache ViTextVQA (W08) — file cache nằm trên Drive, đây là biên bản để tái lập',
       'created_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'), 'caches': []}
for p in [C_PADDLE, C_VIET]:
    if not os.path.exists(p):
        continue
    c = load_ocr_cache(p)
    cover = {k: sum(x in c['images'] for x in v) for k, v in IMGS.items()}
    man['caches'].append({'file': p.replace(DRIVE_ROOT + '/', ''), 'sha256': sha256(p), 'backend': c['backend'],
                          'engine_info': c['engine_info'], 'n_images': len(c['images']),
                          'n_errors': sum(1 for e in c['images'].values() if e.get('error')),
                          'timing': c['timing'], 'images_covered': cover,
                          'images_needed': {k: len(v) for k, v in IMGS.items()}})
    print(f"{c['backend']:18s} {len(c['images']):5d} ảnh | {c['timing']['sec'] / c['timing']['images']:.2f} s/ảnh | "
          f"sha256 {man['caches'][-1]['sha256'][:12]} | đủ: {cover}")
os.makedirs(f'{EXP_ROOT}/W08_ocr_cache', exist_ok=True)
json.dump(man, open(f'{EXP_ROOT}/W08_ocr_cache/ocr_cache_manifest.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('->', f'{EXP_ROOT}/W08_ocr_cache/ocr_cache_manifest.json')